# 🩺 SQL na Saúde
### Fundamentos de SQL para Bancos de Dados em Saúde
**Universidade Estadual da Paraíba (UEPB)** · Base de dados: **Sífilis na Paraíba — DATASUS/SINAN**

---

Este caderno acompanha os slides da aula. Aqui você vai **ler, executar, modificar e praticar** cada comando de SQL usando dados reais de notificação de sífilis dos municípios paraibanos.

> 🧭 **Dica de ouro:** não tenha medo de errar. Altere os números, troque os nomes das colunas, apague uma vírgula... O pior que pode acontecer é aparecer uma mensagem de erro — e ela vai te ensinar algo!

## 🗺️ Como usar este caderno

| Ícone | O que significa |
|:--:|---|
| 📘 | **Explicação** — texto para ler antes do código |
| ▶️ | **Exemplo dos slides** — o mesmo código que aparece na apresentação |
| 🔎 | **Laboratório** — variações do exemplo para você rodar e observar o que muda |
| 🎯 | **Prática** — exercício proposto na aula |
| ✍️ | **Sua vez** — célula em branco para você escrever a sua resposta |
| ✅ | **Resposta** — a solução comentada (confira só depois de tentar!) |
| ⚠️ | **Atenção** — pegadinha ou erro comum |
| 💡 | **Dica** |
| ⭐ | **Bônus** — conteúdo opcional para ir além |

### Passo a passo
1. Clique em **Arquivo → Salvar uma cópia no Drive** para ter a sua versão editável.
2. Execute as células **em ordem**, de cima para baixo: clique na célula e aperte **`Shift + Enter`** (ou no botão ▶).
3. **Comece sempre pela Seção 1.** Sem ela o banco de dados não existe e nenhuma consulta funciona.
4. Se o Colab reiniciar ou desconectar, rode a Seção 1 de novo.
5. Para ver o índice completo, abra o menu lateral (ícone ☰) à esquerda.

## 📑 Índice

1. [⚙️ Configuração do ambiente](#scrollTo=s1_cfg)
2. [🗄️ Conhecendo o banco de dados](#scrollTo=s2_banco)
3. [🔍 Anatomia de uma consulta: SELECT, AS, FROM, WHERE, ORDER BY, LIMIT](#scrollTo=s3_select)
4. [⚖️ Operadores de comparação (+ Prática 1)](#scrollTo=s4_comp)
5. [🎛️ Filtros avançados: AND, OR, IN, LIKE, BETWEEN (+ Prática 2)](#scrollTo=s5_filtros)
6. [📊 Funções de agregação: COUNT, SUM, AVG, MIN, MAX, ROUND, CAST](#scrollTo=s6_agreg)
7. [🧮 GROUP BY (+ Prática 3)](#scrollTo=s7_group)
8. [🔗 Cruzando tabelas: INNER JOIN e LEFT JOIN (+ Prática 4)](#scrollTo=s8_join)
9. [🕳️ Dados ausentes: IS NULL e IS NOT NULL (+ Prática 5)](#scrollTo=s9_nulos)
10. [🐍 Extra: SQL + Python (pandas e gráficos)](#scrollTo=s10_python)
11. [⭐ Bônus: DISTINCT, HAVING, CASE WHEN, COALESCE](#scrollTo=s11_bonus)
12. [🏆 Desafios finais](#scrollTo=s12_desafios)
13. [📌 Cola rápida e erros comuns](#scrollTo=s13_cola)

---
# 1. ⚙️ Configuração do ambiente
*(Slides 4 e 5)*

### 📘 O que vamos fazer aqui?
Antes de consultar qualquer dado, precisamos de **duas coisas**:

1. **Baixar o banco de dados** (o arquivo `sifilis_pb.db`) para dentro do Colab;
2. **Ativar o "poder SQL"** no caderno, para podermos escrever comandos SQL direto nas células.

Isso é feito **uma única vez** por sessão.

### ▶️ Passo 1 — Download do banco de dados e instalação do suporte a SQL

In [ ]:
!wget -O sifilis_pb.db "https://raw.githubusercontent.com/vivianrodrigues-dev/Minicurso-SQL-Saude/main/sifilis_pb.db"
!pip install -q jupysql

#### 📘 Entendendo cada parte

| Trecho | O que faz |
|---|---|
| `!` | O ponto de exclamação avisa ao Colab: *"execute isto como comando do sistema"*, e não como código Python. |
| `wget` | Ferramenta que **baixa um arquivo da internet**. Aqui, ele baixa o banco de dados do GitHub. |
| `-O sifilis_pb.db` | A letra `-O` (de *Output*, saída) define **o nome que o arquivo terá** quando for salvo na sessão do Colab. |
| `"https://raw.github..."` | O **endereço** do arquivo no GitHub (repositório *Banco-de-Dados-Sifilis-PB*). |
| `pip install` | Instala pacotes do Python. |
| `jupysql` | O pacote que nos permite **escrever SQL dentro das células**. |
| `-q` | *quiet* (silencioso): instala sem encher a tela de mensagens. |

### ▶️ Passo 2 — Carregamento da extensão SQL e conexão com o SQLite

In [ ]:
%load_ext sql
%config SqlMagic.autopandas = False
%config SqlMagic.feedback = 0
%sql sqlite:///sifilis_pb.db

#### 📘 Entendendo cada parte

| Trecho | O que faz |
|---|---|
| `%` | Marca um **comando mágico** (*Magic Command*) do Jupyter/Colab: um atalho especial que não é Python puro. |
| `%load_ext sql` | **Liga** o plugin de SQL (jupysql) neste caderno. |
| `%config SqlMagic.autopandas = False` | Os resultados aparecem como **tabela do SQL**, e não como DataFrame do pandas. |
| `%config SqlMagic.feedback = 0` | Desliga mensagens extras (como *"Done"* ou *"X rows affected"*), deixando a tela mais limpa. |
| `%sql sqlite:///sifilis_pb.db` | **Conecta** o plugin ao arquivo `sifilis_pb.db` que baixamos no passo anterior. |

> 🗃️ **Por que SQLite?** O SQLite guarda o banco de dados inteiro em **um único arquivo portátil** (`.db`). Isso é ideal para Unidades Básicas de Saúde (UBS) e Gerências Regionais de Saúde, pois permite análise **mesmo sem internet estável** (estratégia *offline-first*).

### 📘 `%sql` ou `%%sql`? Uma ou duas porcentagens?
| Forma | Quando usar |
|---|---|
| `%sql ...` (uma %) | Para **uma única linha** de SQL. |
| `%%sql` (duas %) | Para **a célula inteira** ser SQL (consultas com várias linhas). **Precisa estar na primeira linha da célula!** |

Em todo este caderno usaremos `%%sql`.

### 🔎 Teste: a conexão funcionou?
Vamos pedir ao banco que liste as tabelas que ele possui:

In [ ]:
%%sql
SELECT name AS tabela
FROM sqlite_master
WHERE type = 'table';

> ✅ Se apareceram `municipios`, `sifilis_notificacoes2`, `sifilis_idade` e `sifilis_clinica2`, está tudo certo!
>
> ⚠️ **Se deu erro:** volte e rode as duas células de configuração, **na ordem**. Se o Colab foi reiniciado, rode tudo de novo.

### 💡 Dica opcional: ver mais linhas
Por padrão, o resultado mostra **no máximo 10 linhas** (você verá a mensagem *"Truncated to displaylimit of 10"*). Para ver mais, rode a célula abaixo:

In [ ]:
# Opcional: aumenta o limite de linhas exibidas (troque 25 pelo número que quiser)
%config SqlMagic.displaylimit = 25

---
# 2. 🗄️ Conhecendo o banco de dados
*(Slides 3 e 6)*

### 📘 O que é um banco de dados relacional?
É um conjunto de **tabelas** (parecidas com planilhas de Excel) que se **relacionam** entre si. Cada tabela guarda um tipo de informação, e cada linha é um **registro**.

Nosso banco vem do **DATASUS/SINAN** e reúne notificações de **sífilis nos municípios da Paraíba**, de **2020 a 2024**.

### 🗂️ As 4 tabelas

| Tabela | O que guarda |
|---|---|
| 🗺️ **`municipios`** *(tabela mestra)* | Cadastro oficial: código IBGE e nome da cidade. |
| 📅 **`sifilis_notificacoes2`** | Notificações **mês a mês** (jan a dez) e total do ano. |
| 👥 **`sifilis_idade`** | Casos por **faixa etária**: `"10-14"`, `"15-19"`, `"20-39"`, `"40-59"`. |
| 🩺 **`sifilis_clinica2`** | **Classificações clínicas**: primária, secundária, terciária, latente e `ign_branco` (ignorado/em branco). |

### 🔑 Chaves: como as tabelas se conectam?
- **PRIMARY KEY (chave primária):** `codigo_municipio` na tabela `municipios`. É o identificador **único** de cada município (não se repete).
- **FOREIGN KEY (chave estrangeira):** a coluna `codigo_municipio` presente nas **outras três tabelas**. Ela aponta de volta para `municipios`, permitindo trocar o código IBGE pelo **nome da cidade**.

```
                    ┌──────────────────────┐
                    │      municipios      │
                    │ 🔑 codigo_municipio  │
                    │    municipio         │
                    └──────────┬───────────┘
                               │ (chave estrangeira em cada tabela abaixo)
        ┌──────────────────────┼──────────────────────┐
        ▼                      ▼                      ▼
┌───────────────────┐ ┌─────────────────┐ ┌──────────────────┐
│sifilis_notificacoes2│ │  sifilis_idade  │ │  sifilis_clinica2│
│ ano, jan … dez,   │ │ ano, "10-14",   │ │ ano, primaria,   │
│ total             │ │ "15-19", …      │ │ secundaria, …    │
└───────────────────┘ └─────────────────┘ └──────────────────┘
```

### 📖 Dicionário de colunas

| Tabela | Colunas |
|---|---|
| `municipios` | `codigo_municipio` (🔑 número IBGE), `municipio` (texto, em **MAIÚSCULAS e sem acento**, ex.: `'JOAO PESSOA'`) |
| `sifilis_notificacoes2` | `ano`, `codigo_municipio`, `jan`, `fev`, `mar`, `abr`, `mai`, `jun`, `jul`, `ago`, `set`, `out`, `nov`, `dez`, `total` |
| `sifilis_idade` | `codigo_municipio`, `ano`, `"10-14"`, `"15-19"`, `"20-39"`, `"40-59"` |
| `sifilis_clinica2` | `codigo_municipio`, `ano`, `ign_branco`, `primaria`, `secundaria`, `terciaria`, `latente`, `total` |

> ⚠️ **Atenção a duas pegadinhas:**
> - Colunas com **hífen ou começando por número** (`"10-14"`, `"20-39"`) precisam estar entre **aspas duplas**.
> - `set` (setembro) também é uma **palavra reservada** do SQL, então use `"set"` entre aspas duplas.

### 🔎 Laboratório: espiando cada tabela
`LIMIT 5` pede só as 5 primeiras linhas, para não lotar a tela.

In [ ]:
%%sql
SELECT * FROM municipios LIMIT 5;

In [ ]:
%%sql
SELECT * FROM sifilis_notificacoes2 LIMIT 5;

In [ ]:
%%sql
SELECT * FROM sifilis_idade LIMIT 5;

In [ ]:
%%sql
SELECT * FROM sifilis_clinica2 LIMIT 5;

### 🔎 Laboratório: quantas linhas tem cada tabela?

In [ ]:
%%sql
SELECT 'municipios' AS tabela, COUNT(*) AS linhas FROM municipios
UNION ALL
SELECT 'sifilis_notificacoes2', COUNT(*) FROM sifilis_notificacoes2
UNION ALL
SELECT 'sifilis_idade', COUNT(*) FROM sifilis_idade
UNION ALL
SELECT 'sifilis_clinica2', COUNT(*) FROM sifilis_clinica2;

> 🤔 **Repare:** `municipios` tem mais linhas do que o número de municípios que aparecem em `sifilis_notificacoes2`. Guarde essa observação, ela será importante na seção de **LEFT JOIN**!

### 🔎 Laboratório: ver a estrutura de uma tabela
O comando `PRAGMA table_info(...)` mostra o nome e o tipo de cada coluna.

In [ ]:
%%sql
PRAGMA table_info(sifilis_clinica2);

---
# 3. 🔍 Anatomia de uma consulta SQL
*(Slide 7)*

### 📘 A frase que resume tudo
Uma consulta SQL é como **fazer uma pergunta ao banco**, sempre em uma ordem fixa:

| Palavra | A pergunta que ela responde |
|---|---|
| **`SELECT`** | **Quais colunas** você quer visualizar? |
| **`AS`** | Qual **apelido** quer dar à coluna na saída? |
| **`FROM`** | De **qual tabela** virão as informações? |
| **`WHERE`** | Qual o **filtro epidemiológico** (condição) aplicado? |
| **`ORDER BY`** | **Como ordenar** o resultado (crescente `ASC` ou decrescente `DESC`)? |
| **`LIMIT`** | **Quantas linhas** mostrar, no máximo? |

> 🧠 **Macete:** leia como uma frase — *"Quero [SELECT] estas colunas, da tabela [FROM] tal, onde [WHERE] valer esta condição, ordenadas por [ORDER BY] tal coluna, só as [LIMIT] primeiras."*

> ⚠️ Cada consulta termina com **ponto e vírgula `;`** e as palavras precisam seguir **esta ordem**: `SELECT → FROM → WHERE → ORDER BY → LIMIT`.

### ▶️ Exemplo do slide: a estrutura de uma consulta

In [ ]:
%%sql
SELECT
    codigo_municipio,
    ano,
    "10-14" AS faixa_10_14,
    "15-19" AS faixa_15_19,
    "20-39" AS faixa_20_39,
    "40-59" AS faixa_40_59
FROM sifilis_idade
WHERE ano = 2023
ORDER BY "20-39" DESC
LIMIT 5;

**Lendo a consulta acima, linha por linha:**
- `SELECT ... "10-14" AS faixa_10_14` → escolhe as colunas e **renomeia** as de faixa etária para nomes mais legíveis.
- `FROM sifilis_idade` → os dados vêm da tabela de idades.
- `WHERE ano = 2023` → só linhas do ano de 2023.
- `ORDER BY "20-39" DESC` → ordena da **maior** para a menor quantidade na faixa de 20 a 39 anos.
- `LIMIT 5` → mostra só as **5 primeiras** (as cidades com mais casos nessa faixa).

### 🔎 Laboratório: construindo a consulta peça por peça
Rode cada célula e **observe o que mudou** em relação à anterior.

**1) `SELECT *` traz todas as colunas** (o `*` significa "tudo"):

In [ ]:
%%sql
SELECT * FROM municipios LIMIT 5;

**2) Escolhendo colunas específicas** (traz só o que interessa):

In [ ]:
%%sql
SELECT municipio FROM municipios LIMIT 5;

**3) Dando apelidos com `AS`** (só muda o nome na saída, o banco continua igual):

In [ ]:
%%sql
SELECT
    codigo_municipio AS codigo_ibge,
    municipio AS nome_da_cidade
FROM municipios
LIMIT 5;

**4) Filtrando com `WHERE`** (só as linhas que atendem à condição):

In [ ]:
%%sql
SELECT ano, codigo_municipio, total
FROM sifilis_notificacoes2
WHERE ano = 2023
LIMIT 5;

**5) Ordenando com `ORDER BY`** — `DESC` = do maior para o menor:

In [ ]:
%%sql
SELECT ano, codigo_municipio, total
FROM sifilis_notificacoes2
WHERE ano = 2023
ORDER BY total DESC
LIMIT 5;

**6) Mudando para `ASC`** (do menor para o maior — é o padrão quando você não escreve nada):

In [ ]:
%%sql
SELECT ano, codigo_municipio, total
FROM sifilis_notificacoes2
WHERE ano = 2023
ORDER BY total ASC
LIMIT 5;

**7) Ordenando por texto** (ordem alfabética):

In [ ]:
%%sql
SELECT municipio
FROM municipios
ORDER BY municipio ASC
LIMIT 10;

> 💬 **Comentários no SQL:** tudo depois de `--` na linha é ignorado pelo banco. Use para anotar o que cada trecho faz (como fazemos neste caderno). Para comentar várias linhas, use `/* ... */`.

---
# 4. ⚖️ Operadores de comparação
*(Slides 8 a 10)*

### 📘 O que são?
São os símbolos usados no `WHERE` para **comparar** o valor de uma coluna com outro valor.

| Operador | Significado | Exemplo |
|:--:|---|---|
| `=` | igual a | `ano = 2023` |
| `<>` | **diferente** de | `codigo_municipio <> 250407` |
| `>` | maior que | `total > 10` |
| `<` | menor que | `total < 500` |
| `>=` | maior **ou igual** a | `ano >= 2020` |
| `<=` | menor **ou igual** a | `ano <= 2023` |

> ⚠️ **Números** vão **sem aspas** (`ano = 2023`). **Textos** vão entre **aspas simples** (`municipio = 'CAMPINA GRANDE'`), em MAIÚSCULAS e **sem acento**, do jeito que estão gravados no banco.

### ▶️ Exemplo do slide: SELECT – FROM – WHERE – comparadores

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE codigo_municipio <> 250407   -- diferente de Campina Grande
  AND ano >= 2020                  -- maior ou igual a 2020
  AND ano <= 2023                  -- menor ou igual a 2023
  AND total > 10                   -- estritamente maior que 10 casos
  AND total < 500;                 -- estritamente menor que 500 casos

**Como ler:** *"Mostre código, ano e total das notificações, **exceto** Campina Grande, de **2020 a 2023**, em que o total de casos seja **maior que 10 e menor que 500**."*

O `AND` significa que **todas** as condições precisam ser verdadeiras ao mesmo tempo.

### 🔎 Laboratório: um operador de cada vez

**Igualdade (`=`) com apelidos** — qual é o código IBGE de Campina Grande?

In [ ]:
%%sql
SELECT
    codigo_municipio AS codigo_ibge,
    municipio AS nome_cidade
FROM municipios
WHERE codigo_municipio = 250407;

**Igualdade com texto** — agora buscando pelo **nome** (aspas simples!). Troque por `'JOAO PESSOA'` ou `'PATOS'` e rode de novo:

In [ ]:
%%sql
SELECT codigo_municipio, municipio
FROM municipios
WHERE municipio = 'CAMPINA GRANDE';

**Diferente (`<>`)** — todos os registros de 2023, exceto os de João Pessoa:

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE ano = 2023
  AND codigo_municipio <> 250810
LIMIT 10;

**Maior ou igual (`>=`) × maior (`>`)** — compare as duas consultas e veja a diferença quando o total é exatamente 20:

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE ano = 2023 AND total >= 20
ORDER BY total ASC;

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE ano = 2023 AND total > 20
ORDER BY total ASC;

### 🎯 Prática 1
> **Enunciado:** Mostre os municípios com código IBGE **maior** que 250800.
>
> 💡 **Dica:** Use `SELECT codigo_municipio, municipio`, `FROM municipios` e o operador `>` no `WHERE`.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Prática 1
-- Apague a linha abaixo e escreva a sua consulta aqui:
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta da Prática 1
> ⚠️ *Tente resolver primeiro! Só depois rode a célula abaixo para conferir.*

In [ ]:
%%sql
-- PRÁTICA 1
SELECT codigo_municipio, municipio
FROM municipios
WHERE codigo_municipio > 250800;

**Explicando:** pedimos o código e o nome (`SELECT`) da tabela `municipios` (`FROM`), mantendo só as linhas em que o código é **maior que 250800** (`WHERE`). Note que o resultado aparece truncado em 10 linhas — são apenas o limite de exibição, não de dados (veja a dica de `displaylimit` na Seção 1).

---
# 5. 🎛️ Filtros avançados
*(Slides 11 a 13)*

### 📘 Combinando condições
Com estes operadores, montamos filtros precisos para isolar **perfis epidemiológicos** específicos:

| Operador | Para que serve | Exemplo |
|---|---|---|
| **`AND`** | **Todas** as condições devem ser verdadeiras | `ano = 2023 AND total > 10` |
| **`OR`** | **Pelo menos uma** condição verdadeira | `jan > 0 OR fev > 0` |
| **`IN (...)`** | O valor está **dentro de uma lista** | `codigo_municipio IN (250407, 250810)` |
| **`LIKE`** | Busca por **parte do texto** | `municipio LIKE 'CAMPINA%'` |
| **`BETWEEN … AND …`** | Valor **dentro de um intervalo** (inclui os extremos) | `ano BETWEEN 2021 AND 2023` |

#### 🔤 Os curingas do `LIKE`
| Símbolo | Significado | Exemplo | Encontra |
|:--:|---|---|---|
| `%` | **qualquer quantidade** de caracteres (inclusive nenhum) | `'M%'` | nomes que **começam** com M |
| `%` | | `'%GRANDE'` | nomes que **terminam** com GRANDE |
| `%` | | `'%SANTA%'` | nomes que **contêm** SANTA em qualquer posição |
| `_` | **exatamente um** caractere | `'_ATOS'` | `PATOS`, `RATOS`... |

### ▶️ Exemplos do slide (um por vez, já funcionando no nosso banco)
**`AND`** — várias condições ao mesmo tempo:

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE ano = 2023 AND total > 10;

**`IN`** — lista de municípios (Campina Grande e João Pessoa):

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE codigo_municipio IN (250407, 250810);

**`LIKE`** — municípios cujo nome **começa** com `CAMPINA`:

In [ ]:
%%sql
SELECT codigo_municipio, municipio
FROM municipios
WHERE municipio LIKE 'CAMPINA%';

**`BETWEEN`** — intervalo de anos (os dois extremos **entram** no resultado):

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE ano BETWEEN 2020 AND 2023
LIMIT 10;

### 🔎 Laboratório: combinando tudo

**`WHERE` + `OR` + `BETWEEN` + `AND`** — Campina Grande *ou* João Pessoa, entre 2021 e 2023, com mais de 50 casos:

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE (codigo_municipio = 250407 OR codigo_municipio = 250810)  -- Campina Grande ou João Pessoa
  AND ano BETWEEN 2021 AND 2023                                 -- intervalo fechado de anos
  AND total > 50;

**`IN` + `OR` + `LIKE`** — municípios de uma lista exata de códigos **ou** com nome começando por `M`:

In [ ]:
%%sql
SELECT codigo_municipio, municipio
FROM municipios
WHERE codigo_municipio IN (250407, 250810, 250100)  -- pertence a esta lista de códigos IBGE
   OR municipio LIKE 'M%';                          -- nomes que começam com a letra M

**`LIKE` com `%` em posições diferentes** — rode as 3 e compare:

In [ ]:
%%sql
-- começa com SAO
SELECT municipio FROM municipios WHERE municipio LIKE 'SAO%';

In [ ]:
%%sql
-- termina com GRANDE
SELECT municipio FROM municipios WHERE municipio LIKE '%GRANDE';

In [ ]:
%%sql
-- contém SANTA
SELECT municipio FROM municipios WHERE municipio LIKE '%SANTA%';

**`NOT IN`** — o contrário do `IN`: todos, **menos** os da lista:

In [ ]:
%%sql
SELECT codigo_municipio, ano, total
FROM sifilis_notificacoes2
WHERE ano = 2023
  AND codigo_municipio NOT IN (250810, 250407)
ORDER BY total DESC
LIMIT 5;

### ⚠️ Cuidado com os parênteses ao misturar `AND` e `OR`!
O `AND` é avaliado **antes** do `OR`. Sem parênteses, o resultado pode **não ser o que você pensa**. Compare as duas consultas (ambas contam linhas):

In [ ]:
%%sql
-- SEM parênteses: lê-se "(João Pessoa E 2023) OU (Campina Grande, de qualquer ano)"
SELECT COUNT(*) AS linhas
FROM sifilis_notificacoes2
WHERE codigo_municipio = 250810 AND ano = 2023 OR codigo_municipio = 250407;

In [ ]:
%%sql
-- COM parênteses: "(João Pessoa OU Campina Grande) E 2023"
SELECT COUNT(*) AS linhas
FROM sifilis_notificacoes2
WHERE (codigo_municipio = 250810 OR codigo_municipio = 250407) AND ano = 2023;

> 💡 **Regra prática:** sempre que usar `OR` junto com `AND`, coloque parênteses para deixar a sua intenção explícita.

### 🎯 Prática 2
> **Enunciado:** Selecione **todos** os registros da tabela `sifilis_notificacoes2` cujo ano seja **2022 ou 2023**, onde o total de casos esteja **entre 20 e 100**, o código do município **comece com '25'** e tenha **pelo menos um caso** registrado no mês de **janeiro ou fevereiro**.
>
> 💡 **Dica:** São 4 condições ligadas por `AND`: `IN` para os anos, `BETWEEN` para o total, `LIKE` para o código e `OR` (entre parênteses) para janeiro/fevereiro.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Prática 2
-- Apague a linha abaixo e escreva a sua consulta aqui:
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta da Prática 2
> ⚠️ *Tente resolver primeiro! Só depois rode a célula abaixo para conferir.*

In [ ]:
%%sql
-- PRÁTICA 2
SELECT *
FROM sifilis_notificacoes2
WHERE ano IN (2022, 2023)
  AND total BETWEEN 20 AND 100
  AND codigo_municipio LIKE '25%'
  AND (jan > 0 OR fev > 0);

-- IN: filtra os anos de interesse (2022 ou 2023).
-- BETWEEN: limita o intervalo da coluna total entre 20 e 100.
-- LIKE: filtra códigos de município que iniciam com o padrão '25%'.
-- OR: verifica a presença de casos em janeiro ou fevereiro (jan > 0 OR fev > 0).
-- AND: garante que todas essas condições sejam satisfeitas simultaneamente.

**Explicando:** cada requisito do enunciado virou uma condição.

| Pedido do enunciado | Trecho em SQL |
|---|---|
| ano 2022 **ou** 2023 | `ano IN (2022, 2023)` |
| total **entre** 20 e 100 | `total BETWEEN 20 AND 100` |
| código **começa com** '25' | `codigo_municipio LIKE '25%'` |
| caso em janeiro **ou** fevereiro | `(jan > 0 OR fev > 0)` |

> 🧠 **Curiosidade:** todos os códigos IBGE da Paraíba começam com `25` (é o código do estado). Por isso, nesta base, esse filtro `LIKE '25%'` não elimina nenhuma linha — mas em uma base com vários estados ele faria toda a diferença.

---
# 6. 📊 Funções de agregação
*(Slide 14)*

### 📘 O que são?
Funções que **resumem várias linhas em um único número** — essenciais para produzir indicadores em saúde pública.

| Função | O que faz | Exemplo em saúde |
|---|---|---|
| **`COUNT()`** | **Conta** quantos registros existem | Quantos anos/registros há por município? |
| **`SUM()`** | **Soma** os valores | Total de casos confirmados |
| **`AVG()`** | Calcula a **média** | Média anual de ocorrências |
| **`MIN()`** | Encontra o **menor** valor | Ano com menos notificações |
| **`MAX()`** | Encontra o **maior** valor | Pico epidemiológico |
| **`ROUND(x, n)`** | **Arredonda** `x` para `n` casas decimais | Média com 2 casas |
| **`CAST(x AS tipo)`** | **Converte** o tipo do dado (ex.: inteiro → decimal) | Evitar divisão inteira |

### 🔎 Laboratório 1: resumindo a tabela inteira (sem agrupar)
Sem `GROUP BY`, as funções resumem **todas as linhas** em **uma só**:

In [ ]:
%%sql
SELECT
    COUNT(*)   AS numero_de_registros,
    SUM(total) AS total_de_casos,
    AVG(total) AS media_por_registro,
    MIN(total) AS menor,
    MAX(total) AS maior
FROM sifilis_notificacoes2;

**Filtrando antes de resumir** — o `WHERE` é aplicado **primeiro**; depois a função resume o que sobrou:

In [ ]:
%%sql
SELECT
    SUM(total) AS casos_2023,
    MAX(total) AS maior_notificacao_2023
FROM sifilis_notificacoes2
WHERE ano = 2023;

### ▶️ Exemplo do slide: resumo por município

In [ ]:
%%sql
SELECT
    codigo_municipio,
    COUNT(ano)   AS total_anos,
    SUM(total)   AS total_acumulado,
    AVG(total)   AS media_bruta,
    MIN(total)   AS menor_ano,
    MAX(total)   AS maior_ano
FROM sifilis_notificacoes2
GROUP BY codigo_municipio
LIMIT 10;

> 📘 Aqui já aparece o `GROUP BY codigo_municipio`: ele pede *"faça esse resumo **para cada município separadamente**"*. Na próxima seção vamos estudá-lo com calma.

### 🔎 Laboratório 2: todas as funções juntas (com `ROUND` e `CAST`)

In [ ]:
%%sql
SELECT
    codigo_municipio,
    COUNT(ano) AS total_anos_registrados,                -- COUNT: contagem de linhas
    SUM(total) AS total_acumulado_casos,                 -- SUM: soma total dos casos
    AVG(total) AS media_bruta_casos,                     -- AVG: média aritmética
    ROUND(AVG(total), 2) AS media_arredondada,           -- ROUND: arredonda para 2 casas decimais
    MIN(total) AS menor_notificacao_anual,               -- MIN: menor valor registrado no período
    MAX(total) AS maior_notificacao_anual,               -- MAX: maior valor registrado no período
    ROUND(AVG(CAST(total AS FLOAT)), 2) AS media_float   -- CAST: converte inteiro para decimal
FROM sifilis_notificacoes2
GROUP BY codigo_municipio
LIMIT 5;

### 🔎 Laboratório 3: por que o `CAST` existe? (a "divisão inteira")
No SQLite, **inteiro ÷ inteiro = inteiro** (as casas decimais somem!). Vamos calcular **que porcentagem dos casos é sífilis primária**:

In [ ]:
%%sql
SELECT
    SUM(primaria) / SUM(total)                      AS errado_divisao_inteira,
    CAST(SUM(primaria) AS FLOAT) / SUM(total)       AS certo_com_cast,
    ROUND(100.0 * SUM(primaria) / SUM(total), 1)    AS percentual_primaria
FROM sifilis_clinica2;

- **1ª coluna:** deu `0` — a divisão inteira "cortou" o resultado.
- **2ª coluna:** com `CAST(... AS FLOAT)`, o número vira decimal e a conta fica correta.
- **3ª coluna:** multiplicar por `100.0` (com ponto decimal) tem o mesmo efeito e já entrega o **percentual**, arredondado com `ROUND`.

### 🔎 Laboratório 4: `COUNT(*)` ou `COUNT(coluna)`?
- `COUNT(*)` conta **todas as linhas**.
- `COUNT(coluna)` conta só as linhas em que aquela coluna **não é vazia (NULL)**.

Em tabelas com dados ausentes, os dois podem dar números diferentes! Veremos isso na Seção 9.

In [ ]:
%%sql
SELECT COUNT(*) AS linhas, COUNT(total) AS linhas_com_total
FROM sifilis_notificacoes2;

---
# 7. 🧮 GROUP BY
*(Slides 15 a 17)*

### 📘 O que o `GROUP BY` faz?
Ele **transforma várias linhas individuais em resumos consolidados por categoria** (por município, faixa etária, ano...).

> 🏥 **Imagine** uma pilha de fichas de notificação. O `GROUP BY município` separa as fichas **em pilhas, uma para cada cidade**; depois a função de agregação (`SUM`, `COUNT`...) **resume cada pilha** em um único número.

### 🏅 Regra de Ouro
> **Toda coluna no `SELECT` que não seja uma função de agregação precisa estar declarada no `GROUP BY`.**

```
SELECT   codigo_municipio,  SUM(total)     ← codigo_municipio NÃO é função  →  entra no GROUP BY
FROM     sifilis_notificacoes2
GROUP BY codigo_municipio                   ← ✔
```

### ▶️ Exemplo do slide: total acumulado de casos por município

In [ ]:
%%sql
SELECT
    codigo_municipio,
    SUM(total) AS acumulado_casos
FROM sifilis_notificacoes2
GROUP BY codigo_municipio
ORDER BY acumulado_casos DESC
LIMIT 5;

**Lendo a consulta:** agrupe por município (`GROUP BY`), some o total de cada um (`SUM`), ordene do maior para o menor (`ORDER BY ... DESC`) e mostre só os 5 primeiros (`LIMIT 5`).

> 🧠 **Repare:** o `ORDER BY` pode usar o **apelido** criado no `SELECT` (`acumulado_casos`).

### 🔎 Laboratório: agrupando de várias formas

**Por ano** — qual a evolução dos casos na Paraíba? *(o `WHERE` pode filtrar antes de agrupar)*

In [ ]:
%%sql
SELECT
    ano,
    SUM(total) AS casos_na_paraiba
FROM sifilis_notificacoes2
GROUP BY ano
ORDER BY ano;

**Por ano, contando municípios** — quantos municípios notificaram em cada ano? *(`COUNT` e `SUM` na mesma consulta)*

In [ ]:
%%sql
SELECT
    ano,
    COUNT(*)   AS municipios_que_notificaram,
    SUM(total) AS casos
FROM sifilis_notificacoes2
GROUP BY ano
ORDER BY ano;

**Por faixa etária** — as colunas de idade com `SUM` (atenção às aspas duplas!):

In [ ]:
%%sql
SELECT
    ano,
    SUM("10-14") AS de_10_a_14,
    SUM("15-19") AS de_15_a_19,
    SUM("20-39") AS de_20_a_39,
    SUM("40-59") AS de_40_a_59
FROM sifilis_idade
GROUP BY ano
ORDER BY ano;

**Por forma clínica** — como se distribuem os casos em cada ano?

In [ ]:
%%sql
SELECT
    ano,
    SUM(primaria)   AS primaria,
    SUM(secundaria) AS secundaria,
    SUM(terciaria)  AS terciaria,
    SUM(latente)    AS latente,
    SUM(ign_branco) AS ignorado_em_branco
FROM sifilis_clinica2
GROUP BY ano
ORDER BY ano;

**Por mês** — em qual mês da série há mais notificações? *(somando cada coluna mensal)*

In [ ]:
%%sql
SELECT
    SUM(jan) AS jan, SUM(fev) AS fev, SUM(mar) AS mar, SUM(abr) AS abr,
    SUM(mai) AS mai, SUM(jun) AS jun, SUM("set") AS "set", SUM(jul) AS jul,
    SUM(ago) AS ago, SUM(out) AS out, SUM(nov) AS nov, SUM(dez) AS dez
FROM sifilis_notificacoes2;

### ⚠️ Cuidado: o SQLite "deixa passar" quando você esquece o GROUP BY de uma coluna
Veja a consulta abaixo. Ela **quebra a Regra de Ouro**: `ano` está no `SELECT`, mas **não** está no `GROUP BY`.

In [ ]:
%%sql
-- ❌ NÃO FAÇA ISTO: 'ano' não está no GROUP BY
SELECT codigo_municipio, ano, SUM(total) AS acumulado
FROM sifilis_notificacoes2
GROUP BY codigo_municipio
LIMIT 5;

> O SQLite **não dá erro**, mas a coluna `ano` mostra **um ano qualquer** (de um dos registros do grupo) — o que não faz sentido, pois a soma vale para **todos os anos juntos**. Outros bancos (PostgreSQL, MySQL, SQL Server) **recusam** a consulta com erro. **Ensine seu olho a conferir a Regra de Ouro!**

### 🎯 Prática 3
> **Enunciado:** Mostre o **total de casos por município**, do **maior para o menor**, só os **5 primeiros**.
>
> 💡 **Dica:** Combine `SUM(total)`, `GROUP BY codigo_municipio`, `ORDER BY ... DESC` e `LIMIT 5`.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Prática 3
-- Apague a linha abaixo e escreva a sua consulta aqui:
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta da Prática 3
> ⚠️ *Tente resolver primeiro! Só depois rode a célula abaixo para conferir.*

In [ ]:
%%sql
-- PRÁTICA 3
SELECT
    codigo_municipio,
    SUM(total) AS total_casos
FROM sifilis_notificacoes2
GROUP BY codigo_municipio
ORDER BY total_casos DESC
LIMIT 5;

-- SUM(total): soma o número total de casos de cada município.
-- GROUP BY codigo_municipio: agrupa os dados por município.
-- ORDER BY total_casos DESC: ordena os totais do maior para o menor.
-- LIMIT 5: mostra apenas os 5 primeiros do ranking.

**Explicando:** é o mesmo modelo do slide anterior, com o apelido `total_casos`.

> 🔁 **Quer ver o nome da cidade em vez do código?** Isso exige um `JOIN` — volte aqui depois da próxima seção, onde refazemos esta consulta com nomes!

---
# 8. 🔗 Cruzando tabelas: INNER JOIN e LEFT JOIN
*(Slides 18 a 20)*

### 📘 Por que precisamos de `JOIN`?
As tabelas de casos só guardam o **código IBGE** (ex.: `250810`). Para saber que esse código é **João Pessoa**, precisamos **cruzar** com a tabela `municipios`, que liga *código ↔ nome*. É isso que o `JOIN` faz: **une linhas de duas tabelas** usando uma coluna em comum (a chave).

| Tipo | O que retorna |
|---|---|
| 🔗 **`INNER JOIN`** | **Apenas correspondências**: só as linhas cujo código existe **nas duas** tabelas. |
| ⬅️ **`LEFT JOIN`** | **Preserva a tabela principal** (a da esquerda): traz **todos** os seus registros, mesmo que **não haja correspondência** na outra tabela (os campos vazios vêm como `NULL`). |

```
 INNER JOIN                       LEFT JOIN
 ┌─────────┐   ┌─────────┐        ┌─────────┐   ┌─────────┐
 │  A    ┌─┴───┴─┐    B  │        │ ███████ ┌─┴───┴─┐   B  │
 │       │███████│       │        │ ███A███ │███████│      │
 │       └─┬───┬─┘       │        │ ███████ └─┬───┬─┘      │
 └─────────┘   └─────────┘        └─────────┘   └─────────┘
  só a interseção ███               tudo de A + interseção
```

### 🧩 Peças de um JOIN
- **`FROM tabela_a AS a`** → a tabela principal, com um **apelido curto** (`a`, `m`, `c`...);
- **`JOIN tabela_b AS b`** → a tabela que queremos juntar;
- **`ON a.chave = b.chave`** → **a regra de ligação** (quais colunas devem ser iguais);
- Use **`apelido.coluna`** (ex.: `m.municipio`) para dizer **de qual tabela** vem cada coluna.

> ⚠️ A coluna `codigo_municipio` existe em **várias tabelas**. Se você escrever só `codigo_municipio`, o SQL não sabe de qual tabela é e dá o erro **"ambiguous column name"**. Sempre prefixe: `m.codigo_municipio`.

### ▶️ Exemplo do slide: `INNER JOIN`

In [ ]:
%%sql
SELECT
    m.municipio AS cidade,
    c.ano,
    c.primaria,
    c.secundaria
FROM sifilis_clinica2 c
INNER JOIN municipios m
    ON c.codigo_municipio = m.codigo_municipio
LIMIT 10;

> 📝 *Nota:* adicionamos `LIMIT 10` para o resultado não ficar enorme (no slide, o código termina em `;` sem limite).

**Lendo a consulta:** *"Pegue as notificações clínicas (`c`), junte com os nomes dos municípios (`m`) onde os códigos coincidem (`ON`), e mostre cidade, ano e formas primária/secundária."*

### ▶️ Exemplo do slide: `LEFT JOIN`

In [ ]:
%%sql
SELECT
    m.municipio,
    s.ano,
    s.total
FROM municipios AS m
LEFT JOIN sifilis_notificacoes2 AS s
    ON m.codigo_municipio = s.codigo_municipio
LIMIT 10;

> 📝 *Nota:* no slide, falta uma vírgula depois de `s.ano`. Aqui já está corrigido (e com `LIMIT 10`).

**Lendo a consulta:** a tabela principal é `municipios` (à esquerda). **Todos** os municípios aparecem — inclusive os que **nunca** tiveram notificação (nesses, `ano` e `total` ficam vazios, `NULL`).

### 🔎 Laboratório: INNER × LEFT — qual a diferença na prática?
Vamos **contar as linhas** de cada tipo de JOIN entre `municipios` e `sifilis_notificacoes2`:

In [ ]:
%%sql
SELECT COUNT(*) AS linhas_inner_join
FROM municipios m
INNER JOIN sifilis_notificacoes2 s ON m.codigo_municipio = s.codigo_municipio;

In [ ]:
%%sql
SELECT COUNT(*) AS linhas_left_join
FROM municipios m
LEFT JOIN sifilis_notificacoes2 s ON m.codigo_municipio = s.codigo_municipio;

O `LEFT JOIN` trouxe **mais linhas**: as extras são os municípios **sem nenhuma notificação** no período. Vamos vê-los:

In [ ]:
%%sql
SELECT m.codigo_municipio, m.municipio, s.ano, s.total
FROM municipios m
LEFT JOIN sifilis_notificacoes2 s ON m.codigo_municipio = s.codigo_municipio
WHERE s.codigo_municipio IS NULL
LIMIT 10;

> 🔎 Repare que `ano` e `total` aparecem **vazios** (`None`/`NULL`): são os **dados ausentes** da próxima seção. Truque clássico: **`LEFT JOIN` + `IS NULL`** encontra quem **não tem correspondência**.

### 🔎 Laboratório: mais exemplos com `INNER JOIN`
**Notificações com o nome da cidade** (a mesma ideia com as outras tabelas):

In [ ]:
%%sql
SELECT
    m.codigo_municipio,
    m.municipio,
    sn.ano,
    sn.total
FROM sifilis_notificacoes2 sn
INNER JOIN municipios m
    ON sn.codigo_municipio = m.codigo_municipio
LIMIT 10;

In [ ]:
%%sql
SELECT
    m.codigo_municipio,
    m.municipio,
    sc.ano,
    sc.primaria,
    sc.secundaria,
    sc.terciaria,
    sc.latente,
    sc.total
FROM sifilis_clinica2 sc
INNER JOIN municipios m
    ON sc.codigo_municipio = m.codigo_municipio
LIMIT 10;

In [ ]:
%%sql
SELECT
    m.codigo_municipio,
    m.municipio,
    si.ano,
    si."10-14",
    si."15-19",
    si."20-39",
    si."40-59"
FROM sifilis_idade si
INNER JOIN municipios m
    ON si.codigo_municipio = m.codigo_municipio
LIMIT 10;

**`JOIN` + `WHERE` + `ORDER BY` + `LIMIT`** — as 5 cidades com mais casos de sífilis **primária** em 2023:

In [ ]:
%%sql
SELECT
    m.municipio AS cidade,
    c.ano,
    c.primaria,
    c.secundaria,
    c.terciaria,
    c.latente,
    c.ign_branco
FROM sifilis_clinica2 c
INNER JOIN municipios m ON c.codigo_municipio = m.codigo_municipio
WHERE c.ano = 2023 AND c.primaria > 0
ORDER BY c.primaria DESC
LIMIT 5;

**Sífilis recente (primária + secundária) em 2022** — aqui `JOIN` é a forma curta de `INNER JOIN` (significam a mesma coisa):

In [ ]:
%%sql
SELECT
    m.municipio,
    c.ano,
    c.primaria,
    c.secundaria
FROM sifilis_clinica2 c
JOIN municipios m ON c.codigo_municipio = m.codigo_municipio
WHERE c.ano = 2022
LIMIT 5;

### 🔎 Laboratório: `JOIN` + `GROUP BY` — refazendo a Prática 3 com **nomes**
Agora o ranking dos 5 municípios com mais casos mostra a **cidade** e não o código:

In [ ]:
%%sql
SELECT
    m.municipio AS cidade,
    SUM(s.total) AS total_casos
FROM sifilis_notificacoes2 s
JOIN municipios m ON s.codigo_municipio = m.codigo_municipio
GROUP BY m.municipio
ORDER BY total_casos DESC
LIMIT 5;

> 🏅 Veja como `m.municipio` aparece no `SELECT` **e** no `GROUP BY` — a Regra de Ouro, outra vez!

### 🎯 Prática 4
> **Enunciado:** **Junte** `sifilis_idade` com `municipios` e mostre o **nome da cidade**, o **ano** e a **faixa de 20 a 39 anos** apenas para o ano de **2023**, **ordenando** da cidade com **mais casos** para a com **menos**.
>
> 💡 **Dica:** Lembre-se: a coluna `"20-39"` precisa de aspas duplas, e use `i.` e `m.` para indicar a tabela de cada coluna.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Prática 4
-- Apague a linha abaixo e escreva a sua consulta aqui:
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta da Prática 4
> ⚠️ *Tente resolver primeiro! Só depois rode a célula abaixo para conferir.*

In [ ]:
%%sql
-- PRÁTICA 4
SELECT
    m.municipio AS cidade,
    i.ano,
    i."20-39" AS faixa_20_39
FROM sifilis_idade i
INNER JOIN municipios m
    ON i.codigo_municipio = m.codigo_municipio
WHERE i.ano = 2023
ORDER BY i."20-39" DESC;

**Explicando:**
- `FROM sifilis_idade i` + `INNER JOIN municipios m ON ...` → cruza as duas tabelas pelo código do município;
- `i."20-39" AS faixa_20_39` → pega a coluna de idade (entre aspas duplas) e dá um apelido;
- `WHERE i.ano = 2023` → só o ano de 2023;
- `ORDER BY i."20-39" DESC` → do maior número de casos para o menor.

---
# 9. 🕳️ Dados ausentes
*(Slides 21 a 23)*

### 📘 Qualidade de dados: fichas em branco
Bancos reais têm **fichas incompletas**. No mundo do SQL, uma célula **vazia/sem valor** é representada por **`NULL`**.

| Situação | Como aparece | Como filtrar |
|---|---|---|
| Campo **vazio** (sem valor) | `NULL` | `IS NULL` |
| Campo **preenchido** | qualquer valor | `IS NOT NULL` |
| Ficha sem classificação clínica *(SINAN)* | contada em `ign_branco` | `ign_branco > 0` |

> 🚫 **NUNCA use `= NULL`!** `NULL` não é um valor — é a **ausência** de valor, então nada é "igual" a ele. O correto é **`IS NULL`** (está vazio) ou **`IS NOT NULL`** (tem algum valor).

> ⚠️ **NULL não é a mesma coisa que zero.** `0` significa *"foi registrado: nenhum caso"*; `NULL` significa *"não sabemos / não foi informado"*. Em saúde pública, confundir os dois muda a interpretação do indicador.

### ▶️ Exemplo do slide: `IS NOT NULL` + `ign_branco`

In [ ]:
%%sql
SELECT
    codigo_municipio,
    ano,
    ign_branco AS fichas_incompletas,
    total
FROM sifilis_clinica2
WHERE ano = 2023
  AND ign_branco IS NOT NULL
  AND ign_branco > 0
ORDER BY ign_branco DESC;

**Lendo a consulta:** *"Mostre, em 2023, os municípios com **fichas incompletas** (forma clínica ignorada/em branco), das **mais** para as **menos** incompletas."*

O `ign_branco` é uma peça-chave da **qualidade da vigilância**: quanto mais fichas sem classificação clínica, menos confiável fica o retrato da epidemia.

### 🔎 Laboratório: o que acontece com `= NULL`?

In [ ]:
%%sql
-- ❌ ERRADO: nunca devolve nada, mesmo que existam valores vazios
SELECT * FROM municipios WHERE municipio = NULL;

In [ ]:
%%sql
-- ✅ CERTO: a forma correta de procurar vazios
SELECT * FROM municipios WHERE municipio IS NULL;

> 🤔 As duas devolvem **zero linhas** aqui porque a tabela `municipios` não tem nenhum nome vazio. A diferença é que a 1ª **jamais** funcionaria, mesmo se houvesse. Para **ver `NULL` de verdade**, precisamos de uma situação em que ele apareça...

### 🔎 Laboratório: onde estão os `NULL` neste banco?
As tabelas deste banco foram criadas com **valor padrão `0`** (`DEFAULT 0`): quando não há casos, o valor guardado é **0**, e não vazio. Vamos conferir contando valores vazios e zeros em `total`:

In [ ]:
%%sql
SELECT
    COUNT(*)                                      AS total_de_linhas,
    SUM(CASE WHEN total IS NULL THEN 1 ELSE 0 END) AS linhas_com_total_vazio,
    SUM(CASE WHEN total = 0 THEN 1 ELSE 0 END)     AS linhas_com_total_zero
FROM sifilis_notificacoes2;

Nenhum `NULL` na tabela original. **Mas** o `LEFT JOIN` *fabrica* `NULL` quando um município não tem registro — foi o que vimos na seção anterior. Aqui está de novo, agora focando em **um ano específico (2023)**:

In [ ]:
%%sql
-- Quais municípios NÃO tiveram nenhuma notificação em 2023?
SELECT m.codigo_municipio, m.municipio
FROM municipios m
LEFT JOIN sifilis_notificacoes2 s
    ON m.codigo_municipio = s.codigo_municipio
   AND s.ano = 2023            -- ⚠️ o ano fica no ON, e não no WHERE!
WHERE s.total IS NULL
ORDER BY m.municipio
LIMIT 10;

> ⚠️ **Pegadinha:** se o filtro `s.ano = 2023` fosse para o `WHERE`, os municípios sem registro seriam eliminados (porque `NULL = 2023` nunca é verdadeiro) e a consulta voltaria a se comportar como `INNER JOIN`. Para **manter** os sem correspondência, a condição do ano vai no **`ON`**.

### 🔎 Laboratório: `COUNT(*)` × `COUNT(coluna)` com dados ausentes
Agora a diferença do `COUNT` fica clara: o `LEFT JOIN` cria `NULL`s, e `COUNT(coluna)` **não conta** os vazios.

In [ ]:
%%sql
SELECT
    COUNT(*)       AS linhas_do_resultado,
    COUNT(s.total) AS linhas_com_total_preenchido
FROM municipios m
LEFT JOIN sifilis_notificacoes2 s
    ON m.codigo_municipio = s.codigo_municipio
   AND s.ano = 2023;

### 🔎 Laboratório: combinando `JOIN` + `ign_branco`
Quais cidades (com nome) têm fichas incompletas? Em qual ano?

In [ ]:
%%sql
SELECT
    m.codigo_municipio,
    m.municipio,
    c.ano,
    c.ign_branco
FROM municipios m
INNER JOIN sifilis_clinica2 c ON m.codigo_municipio = c.codigo_municipio
WHERE c.ign_branco IS NOT NULL
  AND c.ign_branco > 0
ORDER BY c.ign_branco DESC
LIMIT 10;

### 🎯 Prática 5
> **Enunciado:** Escreva um código SQL usando **`IS NOT NULL`** para exibir apenas os registros onde a coluna **`total`** foi preenchida.
>
> 💡 **Dica:** Use `SELECT *` na tabela `sifilis_notificacoes2` e `WHERE total IS NOT NULL`.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Prática 5
-- Apague a linha abaixo e escreva a sua consulta aqui:
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta da Prática 5
> ⚠️ *Tente resolver primeiro! Só depois rode a célula abaixo para conferir.*

In [ ]:
%%sql
-- PRÁTICA 5
SELECT *
FROM sifilis_notificacoes2
WHERE total IS NOT NULL;

**Explicando:** `IS NOT NULL` mantém só as linhas em que `total` tem algum valor.

> 🧠 **Por que voltaram todas as linhas?** Porque, nesta base, `total` **nunca** é vazio (os dados ausentes foram gravados como `0`, como vimos no laboratório). Em uma base com fichas realmente em branco, esse mesmo código **removeria** as linhas incompletas. Para tirar a prova, rode a contagem abaixo:

In [ ]:
%%sql
SELECT
    COUNT(*) AS todas_as_linhas,
    (SELECT COUNT(*) FROM sifilis_notificacoes2 WHERE total IS NOT NULL) AS so_preenchidas;

---
# 10. 🐍 Extra: SQL + Python (pandas e gráficos)
*(Fechamento e exportação)*

### 📘 Para que serve integrar SQL com Python?
O SQL é ótimo para **buscar e resumir** dados; o Python é ótimo para **analisar e visualizar**. Juntos, formam um fluxo completo:

> 🗄️ **Banco (SQLite)** → 🔍 **Consulta SQL** → 🐼 **DataFrame (pandas)** → 📈 **Gráfico (matplotlib)**

O código abaixo executa uma consulta com `JOIN`, `WHERE`, `GROUP BY`, `ORDER BY` e `LIMIT`, e transforma o resultado em um **gráfico de barras** com os 5 municípios com mais casos de sífilis entre 2020 e 2023.

In [ ]:
# Execução da consulta SQL via Python e visualização gráfica
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Estabelece conexão com o banco SQLite local
conn = sqlite3.connect("sifilis_pb.db")

# 2. Consulta SQL estruturada com JOIN, WHERE, GROUP BY, ORDER BY e LIMIT
query = """
SELECT
    m.municipio AS cidade,
    SUM(s.total) AS total_acumulado
FROM sifilis_notificacoes2 s
JOIN municipios m ON s.codigo_municipio = m.codigo_municipio
WHERE s.ano BETWEEN 2020 AND 2023
GROUP BY m.municipio
ORDER BY total_acumulado DESC
LIMIT 5;
"""

# 3. Carrega o resultado diretamente em um DataFrame do Pandas
df_resultado = pd.read_sql_query(query, conn)
conn.close()

# 4. Geração do gráfico de barras para o relatório de gestão
plt.figure(figsize=(9, 4.5))
barras = plt.bar(df_resultado['cidade'], df_resultado['total_acumulado'], color='#008080')

plt.title("Top 5 Municípios com Maior Número de Casos de Sífilis (2020-2023)", fontsize=12, fontweight='bold')
plt.xlabel("Município de Residência", fontsize=10)
plt.ylabel("Total de Casos Notificados", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Inserção dos rótulos de dados no topo de cada barra
for barra in barras:
    yval = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2.0, yval + 3, f'{int(yval)}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

#### 📘 Entendendo cada bloco

| Passo | O que acontece |
|---|---|
| `import sqlite3, pandas, matplotlib` | Carrega as 3 ferramentas: **conexão com o banco**, **tabelas em Python** (DataFrame) e **gráficos**. |
| `sqlite3.connect("sifilis_pb.db")` | Abre o arquivo do banco de dados pelo Python. |
| `query = ` + 3 aspas duplas | Guarda a consulta SQL em um texto (as **3 aspas** permitem escrever em várias linhas). |
| `pd.read_sql_query(query, conn)` | Roda a consulta e **devolve o resultado como DataFrame**. |
| `conn.close()` | Fecha a conexão (boa prática). |
| `plt.bar(...)` | Desenha o **gráfico de barras** (cidade no eixo X, casos no eixo Y). |
| `plt.title / xlabel / ylabel / grid` | Colocam título, nomes dos eixos e linhas de grade. |
| `for barra in barras: plt.text(...)` | Escreve o **valor no topo de cada barra**. |

### 🔎 Laboratório: vendo o DataFrame e testando outra consulta
Rode a célula abaixo e depois **troque o ano** ou o **LIMIT** para ver o gráfico mudar:

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect("sifilis_pb.db")

ano_escolhido = 2023   # 👈 troque o ano (2020 a 2024)

df = pd.read_sql_query(f"""
SELECT
    ano,
    SUM(primaria)   AS primaria,
    SUM(secundaria) AS secundaria,
    SUM(terciaria)  AS terciaria,
    SUM(latente)    AS latente,
    SUM(ign_branco) AS ign_branco
FROM sifilis_clinica2
WHERE ano = {ano_escolhido}
GROUP BY ano
""", conn)
conn.close()

display(df)  # a tabela (DataFrame)

# Gráfico: casos por forma clínica no ano escolhido
formas = ['primaria', 'secundaria', 'terciaria', 'latente', 'ign_branco']
valores = df.loc[0, formas].astype(int)
plt.figure(figsize=(7, 4))
plt.bar(formas, valores, color='#2a9d8f')
plt.title(f"Casos de sífilis por forma clínica — Paraíba, {ano_escolhido}")
plt.ylabel("Casos")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---
# 11. ⭐ Bônus (opcional)
*Comandos que não estão nos slides, mas que são o **próximo passo natural** para quem quer ir além.*

### ⭐ `DISTINCT` — remover repetições
Mostra cada valor **uma única vez**. Quantos anos diferentes existem na base?

In [ ]:
%%sql
SELECT DISTINCT ano
FROM sifilis_notificacoes2
ORDER BY ano;

### ⭐ `HAVING` — filtrar **depois** de agrupar
- `WHERE` filtra **linhas** (antes do `GROUP BY`).
- `HAVING` filtra **grupos** (depois do `GROUP BY`), geralmente usando uma função de agregação.

Quais municípios acumularam **mais de 100 casos** entre 2020 e 2024?

In [ ]:
%%sql
SELECT
    codigo_municipio,
    SUM(total) AS acumulado
FROM sifilis_notificacoes2
GROUP BY codigo_municipio
HAVING SUM(total) > 100
ORDER BY acumulado DESC;

### ⭐ `CASE WHEN` — criar categorias (o "se... então..." do SQL)
Classificar o **porte da notificação** de cada município em 2023:

In [ ]:
%%sql
SELECT
    codigo_municipio,
    total,
    CASE
        WHEN total >= 100 THEN 'Alta'
        WHEN total >= 30  THEN 'Média'
        ELSE 'Baixa'
    END AS nivel_notificacao
FROM sifilis_notificacoes2
WHERE ano = 2023
ORDER BY total DESC
LIMIT 10;

### ⭐ `COALESCE` — trocar `NULL` por um valor
`COALESCE(coluna, valor)` devolve a coluna **ou, se ela for vazia, o valor indicado**. Útil depois de um `LEFT JOIN`, para exibir `0` ou um texto no lugar de `NULL`:

In [ ]:
%%sql
SELECT
    m.municipio,
    COALESCE(s.total, 0) AS casos_2023   -- se for NULL, mostra 0
FROM municipios m
LEFT JOIN sifilis_notificacoes2 s
    ON m.codigo_municipio = s.codigo_municipio
   AND s.ano = 2023
ORDER BY casos_2023 ASC, m.municipio
LIMIT 10;

---
# 12. 🏆 Desafios finais
*Hora de juntar tudo! Cada desafio mistura vários comandos. Tente primeiro, depois confira a resposta.*

### 🏆 Desafio 1
> Liste os municípios cujo nome **começa com a letra `S`** **e** cujo código IBGE é **maior que 251000**.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Desafio 1
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta do Desafio 1
> ⚠️ *Só abra depois de tentar!*

In [ ]:
%%sql
SELECT codigo_municipio, municipio
FROM municipios
WHERE municipio LIKE 'S%'
  AND codigo_municipio > 251000
ORDER BY municipio;

**Comandos usados:** `LIKE`, `AND`, comparador `>`, `ORDER BY`.

### 🏆 Desafio 2
> Mostre o **total de casos de sífilis na Paraíba por ano**, do ano com **mais** casos para o com **menos**.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Desafio 2
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta do Desafio 2
> ⚠️ *Só abra depois de tentar!*

In [ ]:
%%sql
SELECT
    ano,
    SUM(total) AS casos_na_paraiba
FROM sifilis_notificacoes2
GROUP BY ano
ORDER BY casos_na_paraiba DESC;

**Comandos usados:** `SUM`, `GROUP BY`, `ORDER BY ... DESC`. Observe que 2024 tem menos casos — será que o ano está completo na base?

### 🏆 Desafio 3
> Mostre as **3 cidades com mais casos de sífilis primária em 2022**, exibindo o **nome** da cidade.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Desafio 3
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta do Desafio 3
> ⚠️ *Só abra depois de tentar!*

In [ ]:
%%sql
SELECT
    m.municipio AS cidade,
    c.primaria
FROM sifilis_clinica2 c
INNER JOIN municipios m ON c.codigo_municipio = m.codigo_municipio
WHERE c.ano = 2022
ORDER BY c.primaria DESC
LIMIT 3;

**Comandos usados:** `INNER JOIN`, `WHERE`, `ORDER BY ... DESC`, `LIMIT`.

### 🏆 Desafio 4
> Liste os **municípios que NÃO tiveram nenhuma notificação em 2024**, em ordem alfabética (mostre só os 10 primeiros).

In [ ]:
%%sql
-- ✍️ SUA VEZ — Desafio 4
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta do Desafio 4
> ⚠️ *Só abra depois de tentar!*

In [ ]:
%%sql
SELECT m.codigo_municipio, m.municipio
FROM municipios m
LEFT JOIN sifilis_notificacoes2 s
    ON m.codigo_municipio = s.codigo_municipio
   AND s.ano = 2024
WHERE s.total IS NULL
ORDER BY m.municipio
LIMIT 10;

**Comandos usados:** `LEFT JOIN`, `IS NULL`, `ORDER BY`, `LIMIT`. Lembre: o filtro do ano vai no `ON`!

### 🏆 Desafio 5
> Em 2023, para os municípios com **mais de 20 casos** no total clínico, calcule o **percentual de fichas incompletas** (`ign_branco` ÷ `total` × 100), arredondado em 1 casa decimal, e mostre os **5 piores**.

In [ ]:
%%sql
-- ✍️ SUA VEZ — Desafio 5
SELECT 'escreva sua consulta aqui' AS dica;

#### ✅ Resposta do Desafio 5
> ⚠️ *Só abra depois de tentar!*

In [ ]:
%%sql
SELECT
    m.municipio AS cidade,
    c.ign_branco,
    c.total,
    ROUND(100.0 * c.ign_branco / c.total, 1) AS percentual_incompletas
FROM sifilis_clinica2 c
JOIN municipios m ON c.codigo_municipio = m.codigo_municipio
WHERE c.ano = 2023
  AND c.total > 20
ORDER BY percentual_incompletas DESC
LIMIT 5;

**Comandos usados:** `JOIN`, `WHERE`, `ROUND`, cálculo com `100.0` (evita a divisão inteira), `ORDER BY`, `LIMIT`. Este é um **indicador real de qualidade da vigilância** em saúde!

---
# 13. 📌 Cola rápida e erros comuns

### 📌 Cola rápida de comandos

| Comando | Para que serve | Exemplo |
|---|---|---|
| `SELECT` | Escolher colunas | `SELECT municipio` |
| `*` | Todas as colunas | `SELECT *` |
| `AS` | Dar apelido | `total AS casos` |
| `FROM` | Escolher a tabela | `FROM municipios` |
| `WHERE` | Filtrar linhas | `WHERE ano = 2023` |
| `= <> > < >= <=` | Comparar | `total >= 20` |
| `AND` / `OR` | Combinar condições | `ano = 2023 AND total > 10` |
| `IN (...)` | Valor em uma lista | `ano IN (2022, 2023)` |
| `BETWEEN` | Intervalo (inclusivo) | `total BETWEEN 20 AND 100` |
| `LIKE` | Texto por padrão (`%`, `_`) | `municipio LIKE 'M%'` |
| `ORDER BY` | Ordenar (`ASC`/`DESC`) | `ORDER BY total DESC` |
| `LIMIT` | Limitar nº de linhas | `LIMIT 5` |
| `COUNT SUM AVG MIN MAX` | Resumir valores | `SUM(total)` |
| `ROUND(x, n)` | Arredondar | `ROUND(AVG(total), 2)` |
| `CAST(x AS FLOAT)` | Converter para decimal | `CAST(total AS FLOAT)` |
| `GROUP BY` | Agrupar para resumir | `GROUP BY codigo_municipio` |
| `INNER JOIN ... ON` | Só correspondências | `JOIN municipios m ON ...` |
| `LEFT JOIN ... ON` | Mantém a tabela principal | `LEFT JOIN ... ON ...` |
| `IS NULL` / `IS NOT NULL` | Vazio / preenchido | `total IS NOT NULL` |

### 🧱 Ordem obrigatória de escrita
```sql
SELECT   colunas / funções
FROM     tabela
JOIN     outra_tabela ON ...
WHERE    condição (filtra linhas)
GROUP BY colunas sem função
HAVING   condição sobre grupos      -- ⭐ bônus
ORDER BY coluna [ASC | DESC]
LIMIT    número;
```

### 🧯 Erros comuns e como resolver

| Mensagem / sintoma | Causa provável | Como corrigir |
|---|---|---|
| `no such table: ...` | Banco não foi carregado ou nome da tabela errado | Rode a **Seção 1** e confira o nome da tabela |
| `no such column: ...` | Nome da coluna errado | Veja os nomes com `SELECT * FROM tabela LIMIT 3;` |
| `near "...": syntax error` | Faltou vírgula, aspas ou parêntese; palavra fora de ordem | Releia a consulta: **vírgula entre colunas**, mas **não** depois da última |
| `ambiguous column name: codigo_municipio` | A coluna existe nas duas tabelas do `JOIN` | Use o prefixo: `m.codigo_municipio` |
| Colunas `"10-14"` ou `"set"` dão erro | Faltaram as **aspas duplas** | Escreva `"10-14"` e `"set"` |
| Texto não encontra nada (`'Joao Pessoa'`) | O banco guarda em **MAIÚSCULAS e sem acento** | Use `'JOAO PESSOA'` ou `LIKE` |
| Número vira `0` em uma divisão | **Divisão inteira** | Use `100.0 *` ou `CAST(... AS FLOAT)` |
| `= NULL` não retorna nada | `NULL` não se compara com `=` | Use `IS NULL` / `IS NOT NULL` |
| `LEFT JOIN` agiu como `INNER JOIN` | Filtro da tabela da direita ficou no `WHERE` | Mova a condição para o `ON` |
| Resultado "cortado" em 10 linhas | Limite de exibição do jupysql | `%config SqlMagic.displaylimit = 50` |
| `%%sql` dá erro de sintaxe | `%%sql` **não** estava na **primeira linha** da célula | Deixe-o como a primeira linha, sem nada acima |
| Nada funciona depois de um tempo parado | A sessão do Colab reiniciou | Rode a **Seção 1** novamente |

---
### 🎉 Parabéns! Você já sabe consultar, filtrar, resumir, cruzar e limpar dados em SQL.
**Próximos passos sugeridos:** repita os desafios mudando municípios e anos, invente suas próprias perguntas de vigilância em saúde e tente respondê-las com SQL.